# TP1 MFA RBAC ABAC MongoDB
Notebook pédagogique fourni avec les TP.


Objectif: prototyper le moteur de décision d’accès avec RBAC, ABAC, MFA et journalisation.

In [4]:
import pandas as pd, hashlib, hmac, time, random
users = pd.read_csv('datasets/users.csv')
resources = pd.read_csv('datasets/resources.csv')
users.head(), resources.head()


(  user_id       name            role   department  mfa_enabled  \
 0    u001     Dr Ali         medecin  cardiologie         True   
 1    u002  Mme Houda      secretaire      accueil         True   
 2    u003       Anis       infirmier      urgence         True   
 3    u004      Jamal       infirmier    pediatrie        False   
 4    u005  Admin Sec  admin_securite           it         True   
 
          clearance  
 0          medical  
 1            admin  
 2  medical_limited  
 3  medical_limited  
 4         security  ,
   resource_id patient_id             type owner_department sensitivity
 0        r001       p012  dossier_medical        pediatrie      public
 1        r002       p003    dossier_admin      cardiologie     interne
 2        r003       p001  dossier_medical      cardiologie    sensible
 3        r004       p004  dossier_medical      cardiologie     interne
 4        r005       p004    dossier_admin          accueil    sensible)

In [5]:
# Simulation TOTP simplifiée pour le TP: en production, utiliser pyotp et un secret par utilisateur.
def make_otp(user_id, window=None):
    window = int(time.time()//30) if window is None else window
    digest = hmac.new(user_id.encode(), str(window).encode(), hashlib.sha1).hexdigest()
    return digest[-6:]

def verify_otp(user_id, code):
    w = int(time.time()//30)
    return code in [make_otp(user_id,w-1), make_otp(user_id,w), make_otp(user_id,w+1)]
print(make_otp('u001'))


5d5dc2


In [6]:
POLICY = {
    'medecin': {'read':['dossier_medical','resultat_labo'], 'write':['dossier_medical'], 'export':[]},
    'infirmier': {'read':['dossier_medical','resultat_labo'], 'write':[], 'export':[]},
    'secretaire': {'read':['dossier_admin'], 'write':['dossier_admin'], 'create':['dossier_admin'], 'export':[]},
    'admin_securite': {'read':['journal_acces'], 'write':[], 'export':['journal_acces']},
    'patient': {'read':['dossier_admin'], 'write':[], 'export':[]}
}

def decide_access(user, resource, action, context):
    if not context.get('mfa_ok', False) and resource['sensitivity'] in ['confidentiel','sensible']:
        return False, 'MFA obligatoire pour donnée confidentielle ou sensible'
    if action not in POLICY.get(user['role'], {}):
        return False, 'Action absente de la politique'
    if resource['type'] not in POLICY[user['role']].get(action, []):
        return False, 'Rôle non autorisé sur ce type de ressource'
    if user['role'] in ['medecin','infirmier'] and user['department'] != resource['owner_department']:
        return False, 'Contrainte ABAC: département différent'
    if context.get('hour', 12) < 6 or context.get('hour', 12) > 20:
        return False, 'Contrainte temporelle: hors plage de service'
    return True, 'Accès autorisé'

u = users.iloc[0].to_dict(); r = resources.iloc[0].to_dict()
decide_access(u, r, 'read', {'mfa_ok': True, 'hour': 10})


(False, 'Contrainte ABAC: département différent')

In [7]:
# Journal d'audit minimal
import json
trial = []
for i in range(20):
    u = users.sample(1).iloc[0].to_dict(); r = resources.sample(1).iloc[0].to_dict()
    action = random.choice(['read','write','create','export'])
    ok, reason = decide_access(u,r,action,{'mfa_ok':bool(random.getrandbits(1)),'hour':random.randint(0,23)})
    trial.append({'user_id':u['user_id'],'role':u['role'],'resource_id':r['resource_id'],'action':action,'decision':ok,'reason':reason})
pd.DataFrame(trial).head(10)


,user_id,role,resource_id,action,decision,reason
0,u005,admin_securite,r027,create,False,Action absente de la politique
1,u002,secretaire,r010,create,False,Rôle non autorisé sur ce type de ressource
2,u003,infirmier,r017,read,True,Accès autorisé
3,u006,patient,r008,create,False,Action absente de la politique
4,u004,infirmier,r022,create,False,Action absente de la politique
5,u001,medecin,r039,export,False,MFA obligatoire pour donnée confidentielle ou ...
6,u006,patient,r005,read,True,Accès autorisé
7,u006,patient,r021,write,False,Rôle non autorisé sur ce type de ressource
8,u001,medecin,r013,export,False,Rôle non autorisé sur ce type de ressource
9,u001,medecin,r024,read,False,MFA obligatoire pour donnée confidentielle ou ...


In [9]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Chargement des données
logs = pd.read_csv('datasets/access_logs.csv')
print("Shape :", logs.shape)
print("\nSuccès / Échecs :")
print(logs['success'].value_counts(normalize=True))

# Conversion timestamp
logs['timestamp'] = pd.to_datetime(logs['timestamp'])
logs['hour'] = logs['timestamp'].dt.hour
logs['date'] = logs['timestamp'].dt.date

Shape : (600, 12)

Succès / Échecs :
success
True     0.815
False    0.185
Name: proportion, dtype: float64


In [10]:
# 1. Taux de refus par rôle
refus_par_role = logs.groupby('role')['success'].value_counts(normalize=True).unstack().fillna(0)
print(refus_par_role)

# 2. Refus par raison
print("\nTop raisons de refus :")
print(logs[logs['success'] == False]['reason'].value_counts().head(10))

# 3. Tentatives hors plage horaire
hors_plage = logs[logs['hour'] < 6].shape[0] + logs[logs['hour'] > 20].shape[0]
print(f"\nTentatives hors plage (6h-20h) : {hors_plage}")

# 4. Échecs MFA
echecs_mfa = logs[logs['reason'].str.contains("MFA", na=False)]
print(f"Échecs MFA : {len(echecs_mfa)}")

success            False     True 
role                              
admin_securite  0.191919  0.808081
infirmier       0.171429  0.828571
medecin         0.244444  0.755556
patient         0.188119  0.811881
secretaire      0.150000  0.850000

Top raisons de refus :


KeyError: 'reason'